## Ingestion, Nettoyage, et Parsing

In [10]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Récupération de la page
url = "https://en.wikipedia.org/wiki/Madagascar"
headers = {
    'User-Agent': 'RAG_Madagascar_Project'
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

# On cible le contenu principal pour éviter les menus de navigation
content_div = soup.find(id="mw-content-text").find(class_="mw-parser-output")


In [12]:
# 2. Nettoyage préliminaire du DOM (Suppression du "bruit")
# Supprime les références type [1], [2], [a]
for sup in content_div.find_all("sup", class_="reference"):
    sup.decompose()
    
# Supprime les liens "[edit]" ou "[modifier]"
for span in content_div.find_all("span", class_="mw-editsection"):
    span.decompose()


In [41]:

import io

# 2. Parsing robuste par balise
documents = []
current_h2 = "Introduction"
current_h3 = ""
TABLE_CHUNK_SIZE = 10
# Sections qui marquent la fin de l'article principal à ignorer
STOP_SECTIONS = ["see also", "notes", "references", "external links"]

# On utilise find_all pour récupérer les éléments dans l'ordre où ils apparaissent dans le HTML
# Cela évite de se perdre dans les divs imbriquées
for element in soup.find_all(['h2', 'h3', 'p', 'table']):
    
    # --- HIÉRARCHIE ---
    if element.name == 'h2':
        # On évite le menu de navigation caché
        text = element.get_text(strip=True)
        # Nettoyage du texte du header (parfois il y a un span "[edit]" résiduel)
        clean_header = re.sub(r'\[.*?\]', '', text).strip()
        
        # CONDITION D'ARRÊT : Si on arrive à "See also" ou après, on stoppe la boucle
        if clean_header.lower() in STOP_SECTIONS:
            print(f"Arrêt de l'ingestion à la section : {clean_header}")
            break
        if text and text not in ["Contents", "Navigation menu", "Personal tools"]:
            current_h2 = text
            current_h3 = "" 
            
    elif element.name == 'h3':
        current_h3 = element.get_text(strip=True)

    # --- TEXTE ---
    elif element.name == 'p':
        text = element.get_text(strip=True)
        # On ignore les paragraphes vides ou les phrases d'avertissement Wikipédia
        if len(text) > 50 and not text.startswith("This article is about"):
            metadata = {
                "source": url,
                "h2_section": current_h2,
                "h3_section": current_h3,
                "content_type": "text"
            }
            documents.append(Document(page_content=text, metadata=metadata))

    # --- TABLEAUX ---
    elif element.name == 'table':
        try:
            # 1. Forcer Pandas à lire la chaîne comme un flux de texte en mémoire
            html_string = str(element)
            df = pd.read_html(io.StringIO(html_string))[0]
            
            # 2. Aplatir les MultiIndex si présents
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = ['_'.join(str(c) for c in col if c).strip() for col in df.columns]
            
            headers_list = df.columns.tolist()
            
            for i in range(0, len(df), TABLE_CHUNK_SIZE):
                chunk_df = df.iloc[i : i + TABLE_CHUNK_SIZE]
                chunk_lines = []
                
                for _, row in chunk_df.iterrows():
                    row_elements = []
                    for col in headers_list:
                        val = str(row[col])
                        # Ignorer les valeurs vides, NaN ou les lignes de séparation Wikipédia
                        if val and val.lower() not in ['nan', 'none']:
                            clean_col = re.sub(r'\[.*?\]', '', str(col)).strip()
                            clean_val = re.sub(r'\[.*?\]', '', val).strip()
                            row_elements.append(f"{clean_col}: {clean_val}")
                    
                    if row_elements: # Ajouter seulement si la ligne n'est pas vide
                        chunk_lines.append(" | ".join(row_elements))
                
                if chunk_lines:
                    table_content = "Extrait de tableau:\n" + "\n".join(chunk_lines)
                    
                    metadata = {
                        "source": url,
                        "h2_section": current_h2,
                        "h3_section": current_h3,
                        "content_type": "table"
                    }
                    documents.append(Document(page_content=table_content, metadata=metadata))
                
        except ValueError as ve:
            # Pandas lève une ValueError s'il ne trouve aucun tableau exploitable
            pass
        except Exception as e:
            print(f"Erreur tableau ignorée dans {current_h2}: {e}")

# Vérification : Affichez les 3 premiers documents extraits
for doc in documents[:3]:
    print(f"--- SECTION: {doc.metadata['h2_section']} ---")
    print(doc.page_content[:150] + "...\n")

Arrêt de l'ingestion à la section : See also
--- SECTION: Introduction ---
Extrait de tableau:
Republic of MadagascarRepoblikan'i Madagasikara (Malagasy) République de Madagascar (French): Flag Emblem | Republic of Madagascar...

--- SECTION: Introduction ---
Extrait de tableau:
Republic of MadagascarRepoblikan'i Madagasikara (Malagasy) République de Madagascar (French): • President | Republic of Madagascar...

--- SECTION: Introduction ---
Extrait de tableau:
Republic of MadagascarRepoblikan'i Madagasikara (Malagasy) République de Madagascar (French): • Republic proclaimed | Republic of ...



In [43]:
tables = []
for doc in documents:
    if doc.metadata.get('content_type')=='table':
        
        tables.append(doc)

In [44]:
len(tables)

10

In [45]:
print(tables[5].page_content)

Extrait de tableau:
Number on map: 1 | Region's name: Diana | Area (km2): 19266 | Population (2018 census): 889736 | Population density per km2: 36.3 | Capital: Antsiranana | Former province: Antsiranana
Number on map: 2 | Region's name: Sava | Area (km2): 25518 | Population (2018 census): 1123013 | Population density per km2: 38.4 | Capital: Sambava | Former province: Antsiranana
Number on map: 3 | Region's name: Itasy | Area (km2): 6993 | Population (2018 census): 897962 | Population density per km2: 104.8 | Capital: Miarinarivo | Former province: Antananarivo
Number on map: 4 | Region's name: Analamanga | Area (km2): 16911 | Population (2018 census): 3618128 | Population density per km2: 198.0 | Capital: Antananarivo | Former province: Antananarivo
Number on map: 5 | Region's name: Vakinankaratra | Area (km2): 16599 | Population (2018 census): 2074358 | Population density per km2: 108.6 | Capital: Antsirabe | Former province: Antananarivo
Number on map: 6 | Region's name: Bongolava 

In [50]:
print(tables[9].page_content)

Extrait de tableau:
vte Largest cities or towns in Madagascar According to the 2018 Census_Rank: 1 | vte Largest cities or towns in Madagascar According to the 2018 Census_Name: Antananarivo | vte Largest cities or towns in Madagascar According to the 2018 Census_Region: Analamanga | vte Largest cities or towns in Madagascar According to the 2018 Census_Pop.: 1275207
vte Largest cities or towns in Madagascar According to the 2018 Census_Rank: 2 | vte Largest cities or towns in Madagascar According to the 2018 Census_Name: Toamasina | vte Largest cities or towns in Madagascar According to the 2018 Census_Region: Atsinanana | vte Largest cities or towns in Madagascar According to the 2018 Census_Pop.: 326286
vte Largest cities or towns in Madagascar According to the 2018 Census_Rank: 3 | vte Largest cities or towns in Madagascar According to the 2018 Census_Name: Antsirabe | vte Largest cities or towns in Madagascar According to the 2018 Census_Region: Vakinankaratra | vte Largest cities

In [47]:
print(documents[23].page_content)

Mahafaly tradition has Olembetsitoto as the first Maroseranasacred rulerin the 16th century, who was protected by anombiasy[fr](priest).Prior to 1600, only the north of the island was integrated into Indian Ocean trade; but in the mid-to late-16th century, European merchants (vazaha; "foreigners") began using the newly-namedSt. Augustine Bayin the southwest as a stopping point and traded with the communities there. The British attempted to found a colony at the Bay in 1645 but were expelled by the Malagasy.


In [48]:
print(len(documents))

160


## Chunking du texte long

In [51]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

final_documents = []
for doc in documents:
    if doc.metadata["content_type"] == "text":
        # Split le texte, les métadonnées sont automatiquement dupliquées pour chaque chunk
        chunks = text_splitter.split_documents([doc])
        final_documents.extend(chunks)
    else:
        # Les tableaux sont déjà chunkés correctement (10 lignes), on ne les touche pas
        final_documents.append(doc)

print(f"Nombre total de chunks prêts pour Qdrant : {len(final_documents)}")

Nombre total de chunks prêts pour Qdrant : 170
